# Life Science Notebook: Survival Analysis with Right Censoring (Expanded)

This notebook provides a review-ready time-to-event analysis pipeline with censored outcomes.

What is included:
- synthetic cohort simulation with treatment effect and censoring,
- Kaplan-Meier curves and restricted mean survival time (RMST),
- log-rank test,
- Cox proportional hazards model (via partial likelihood),
- discrete-time hazard model for patient-level dynamic risk,
- subgroup consistency checks.

## 0) Imports and setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.optimize import minimize
from scipy.stats import chi2
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

np.random.seed(321)
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 50)

## 1) Simulate right-censored trial follow-up

In [ ]:
n = 4200
max_follow = 30  # months

df = pd.DataFrame(
    {
        "age": np.random.randint(30, 86, size=n),
        "sex": np.random.choice(["F", "M"], size=n),
        "biomarker": np.random.normal(0.0, 1.0, size=n),
        "comorbidity": np.random.poisson(1.5, size=n),
        "baseline_severity": np.clip(np.random.normal(5.0, 1.4, size=n), 0.2, 10.0),
        "region": np.random.choice(["NA", "EU", "APAC", "LATAM"], p=[0.40, 0.30, 0.20, 0.10], size=n),
        "arm": np.random.choice(["control", "treatment"], p=[0.5, 0.5], size=n),
    }
)

df["elderly"] = (df["age"] >= 65).astype(int)

# proportional hazards data-generating mechanism
linpred = (
    0.017 * (df["age"].to_numpy() - 55)
    + 0.34 * df["biomarker"].to_numpy()
    + 0.23 * df["comorbidity"].to_numpy()
    + 0.11 * df["baseline_severity"].to_numpy()
    - 0.42 * (df["arm"].to_numpy() == "treatment")
    + np.where(df["region"].to_numpy() == "LATAM", 0.09, 0.0)
)

base_hazard = 0.03
hazard = base_hazard * np.exp(linpred)

true_event_time = np.random.exponential(scale=1 / hazard)
loss_to_followup = np.random.uniform(6, max_follow, size=n)

obs_time = np.minimum(true_event_time, loss_to_followup)
event = (true_event_time <= loss_to_followup).astype(int)

df["time"] = np.clip(np.ceil(obs_time), 1, max_follow).astype(int)
df["event"] = event

df.head()

In [ ]:
summary = pd.Series(
    {
        "n_patients": len(df),
        "event_rate": df["event"].mean(),
        "median_observed_time": df["time"].median(),
        "treatment_share": (df["arm"] == "treatment").mean(),
    }
)
summary

## 2) Kaplan-Meier estimator

In [ ]:
def kaplan_meier(times: np.ndarray, events: np.ndarray, t_max: int):
    km = []
    s = 1.0
    at_risk = len(times)

    for t in range(1, t_max + 1):
        d_t = int(((times == t) & (events == 1)).sum())
        c_t = int(((times == t) & (events == 0)).sum())

        if at_risk > 0:
            s *= (1 - d_t / at_risk)

        km.append({"t": t, "survival": s, "n_risk": at_risk, "d": d_t, "c": c_t})
        at_risk -= (d_t + c_t)

    return pd.DataFrame(km)

km_ctrl = kaplan_meier(
    df.loc[df["arm"] == "control", "time"].to_numpy(),
    df.loc[df["arm"] == "control", "event"].to_numpy(),
    max_follow,
)

km_trt = kaplan_meier(
    df.loc[df["arm"] == "treatment", "time"].to_numpy(),
    df.loc[df["arm"] == "treatment", "event"].to_numpy(),
    max_follow,
)

km_ctrl.head(), km_trt.head()

In [ ]:
plt.figure(figsize=(8.5, 4.8))
plt.step(km_ctrl["t"], km_ctrl["survival"], where="post", label="Control", color="#C44E52")
plt.step(km_trt["t"], km_trt["survival"], where="post", label="Treatment", color="#55A868")
plt.title("Kaplan-Meier Curves")
plt.xlabel("Months")
plt.ylabel("Survival probability")
plt.legend()
plt.tight_layout()
plt.show()

## 3) RMST (restricted mean survival time) up to study horizon

In [ ]:
def rmst_from_km(km_df: pd.DataFrame, tau: int):
    # Piecewise-constant survival integration
    times = np.concatenate(([0], km_df["t"].to_numpy()))
    surv = np.concatenate(([1.0], km_df["survival"].to_numpy()))

    area = 0.0
    for i in range(len(times) - 1):
        dt = min(times[i + 1], tau) - times[i]
        if dt > 0:
            area += surv[i] * dt
    return area

rmst_ctrl = rmst_from_km(km_ctrl, max_follow)
rmst_trt = rmst_from_km(km_trt, max_follow)
rmst_diff = rmst_trt - rmst_ctrl

pd.Series({"rmst_control": rmst_ctrl, "rmst_treatment": rmst_trt, "rmst_diff_treatment_minus_control": rmst_diff})

## 4) Log-rank test

In [ ]:
def logrank_test(data: pd.DataFrame, group_col="arm", time_col="time", event_col="event"):
    gvals = data[group_col].unique()
    g1, g2 = gvals[0], gvals[1]
    x = data[data[group_col] == g1]
    y = data[data[group_col] == g2]

    O1 = E1 = V1 = 0.0

    for t in sorted(data[time_col].unique()):
        n1 = (x[time_col] >= t).sum()
        n2 = (y[time_col] >= t).sum()
        n = n1 + n2
        if n <= 1:
            continue

        d1 = ((x[time_col] == t) & (x[event_col] == 1)).sum()
        d2 = ((y[time_col] == t) & (y[event_col] == 1)).sum()
        d = d1 + d2
        if d == 0:
            continue

        e1 = d * (n1 / n)
        v1 = (n1 * n2 * d * (n - d)) / (n**2 * (n - 1)) if n > 1 else 0.0

        O1 += d1
        E1 += e1
        V1 += v1

    chi_sq = ((O1 - E1) ** 2) / max(V1, 1e-12)
    p_val = 1 - chi2.cdf(chi_sq, df=1)
    return chi_sq, p_val

logrank_chi2, logrank_p = logrank_test(df)
pd.Series({"logrank_chi2": logrank_chi2, "p_value": logrank_p})

## 5) Cox proportional hazards model (manual partial likelihood)

In [ ]:
cox_df = df.copy()
cox_df["arm_treat"] = (cox_df["arm"] == "treatment").astype(int)
cox_df["sex_m"] = (cox_df["sex"] == "M").astype(int)

cox_features = ["arm_treat", "age", "biomarker", "comorbidity", "baseline_severity", "elderly", "sex_m"]
X = cox_df[cox_features].to_numpy(dtype=float)
T = cox_df["time"].to_numpy(dtype=float)
E = cox_df["event"].to_numpy(dtype=int)

# Standardize continuous columns for numerical stability
cont_idx = [1, 2, 3, 4]
X_std = X.copy()
for j in cont_idx:
    mu = X[:, j].mean()
    sd = X[:, j].std(ddof=1)
    X_std[:, j] = (X[:, j] - mu) / (sd + 1e-12)

order = np.argsort(T)
Xo = X_std[order]
To = T[order]
Eo = E[order]


def neg_log_partial_lik(beta):
    xb = Xo @ beta
    nll = 0.0
    for i in range(len(To)):
        if Eo[i] == 1:
            risk = To >= To[i]
            denom = np.log(np.sum(np.exp(xb[risk])))
            nll -= xb[i] - denom
    return nll

init = np.zeros(Xo.shape[1])
res = minimize(neg_log_partial_lik, init, method="BFGS")
beta_hat = res.x

hazard_ratios = pd.DataFrame({
    "feature": cox_features,
    "beta": beta_hat,
    "hazard_ratio": np.exp(beta_hat),
}).sort_values("hazard_ratio", ascending=False)

hazard_ratios

## 6) Discrete-time hazard model (person-period format)

In [ ]:
rows = []
for _, r in df.iterrows():
    for t in range(1, int(r["time"]) + 1):
        rows.append(
            {
                "age": r["age"],
                "sex": r["sex"],
                "biomarker": r["biomarker"],
                "comorbidity": r["comorbidity"],
                "baseline_severity": r["baseline_severity"],
                "elderly": r["elderly"],
                "region": r["region"],
                "arm": r["arm"],
                "time_bin": t,
                "event_t": int((t == r["time"]) and (r["event"] == 1)),
            }
        )

pp = pd.DataFrame(rows)
pp.head()

In [ ]:
haz_features = [
    "age",
    "sex",
    "biomarker",
    "comorbidity",
    "baseline_severity",
    "elderly",
    "region",
    "arm",
    "time_bin",
]

Xh = pp[haz_features]
yh = pp["event_t"]

num_cols = ["age", "biomarker", "comorbidity", "baseline_severity", "elderly", "time_bin"]
cat_cols = ["sex", "region", "arm"]

X_train, X_test, y_train, y_test = train_test_split(
    Xh, yh, test_size=0.30, random_state=321, stratify=yh
)

haz_model = Pipeline(
    steps=[
        (
            "prep",
            ColumnTransformer(
                [
                    ("num", StandardScaler(), num_cols),
                    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
                ]
            ),
        ),
        ("clf", LogisticRegression(max_iter=500, random_state=321)),
    ]
)

haz_model.fit(X_train, y_train)
phat = haz_model.predict_proba(X_test)[:, 1]

pd.Series(
    {
        "event_prevalence_person_period": yh.mean(),
        "hazard_model_auc": roc_auc_score(y_test, phat),
    }
)

## 7) Dynamic risk trajectories by arm

In [ ]:
# Predict average hazard by month for each arm using average patient profile
profile_base = {
    "age": float(df["age"].mean()),
    "sex": "F",
    "biomarker": float(df["biomarker"].mean()),
    "comorbidity": float(df["comorbidity"].mean()),
    "baseline_severity": float(df["baseline_severity"].mean()),
    "elderly": 0,
    "region": "NA",
}

traj_rows = []
for arm in ["control", "treatment"]:
    for t in range(1, max_follow + 1):
        row = profile_base.copy()
        row.update({"arm": arm, "time_bin": t})
        q = pd.DataFrame([row])
        hz = haz_model.predict_proba(q[haz_features])[:, 1][0]
        traj_rows.append({"arm": arm, "month": t, "pred_hazard": hz})

traj = pd.DataFrame(traj_rows)

plt.figure(figsize=(8.5, 4.5))
sns.lineplot(data=traj, x="month", y="pred_hazard", hue="arm", marker="o")
plt.title("Predicted Monthly Hazard Trajectory")
plt.xlabel("Month")
plt.ylabel("Predicted hazard")
plt.tight_layout()
plt.show()

## 8) Subgroup consistency checks

In [ ]:
def rmst_by_subset(data: pd.DataFrame, label: str):
    k_ctrl = kaplan_meier(
        data.loc[data["arm"] == "control", "time"].to_numpy(),
        data.loc[data["arm"] == "control", "event"].to_numpy(),
        max_follow,
    )
    k_trt = kaplan_meier(
        data.loc[data["arm"] == "treatment", "time"].to_numpy(),
        data.loc[data["arm"] == "treatment", "event"].to_numpy(),
        max_follow,
    )
    return {
        "subset": label,
        "n": len(data),
        "rmst_control": rmst_from_km(k_ctrl, max_follow),
        "rmst_treatment": rmst_from_km(k_trt, max_follow),
        "rmst_diff": rmst_from_km(k_trt, max_follow) - rmst_from_km(k_ctrl, max_follow),
    }

subsets = []
for grp in ["F", "M"]:
    subsets.append(rmst_by_subset(df[df["sex"] == grp], f"sex={grp}"))

for grp in [0, 1]:
    subsets.append(rmst_by_subset(df[df["elderly"] == grp], f"elderly={grp}"))

subgroup_rmst = pd.DataFrame(subsets)
subgroup_rmst

In [ ]:
plt.figure(figsize=(7.8, 4.5))
sns.barplot(data=subgroup_rmst, x="subset", y="rmst_diff", palette="mako")
plt.axhline(0.0, linestyle="--", color="black")
plt.title("Treatment RMST Gain Across Key Subgroups")
plt.xlabel("Subgroup")
plt.ylabel("RMST difference (treatment - control)")
plt.tight_layout()
plt.show()

## 9) Final summary

- KM, RMST, and log-rank provide interpretable trial-level treatment effect evidence.
- Cox PH offers covariate-adjusted hazard-ratio interpretation.
- Discrete-time hazard modeling enables patient-level dynamic risk forecasting.
- Subgroup RMST checks help surface heterogeneity before policy or protocol decisions.